# Cost-Optimized Model Cascade | Cascading Agents (Model Cascading)

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import TypedDict, Literal
from typing_extensions import NotRequired
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke
import json
import re

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
# Model tiers (cheapest to most expensive)
tier1_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
tier2_model = ChatOpenAI(model="gpt-4o", temperature=0)

class CascadeState(TypedDict):
    query: str
    tier1_response: NotRequired[str]
    tier1_confidence: NotRequired[float]
    tier2_response: NotRequired[str]
    final_response: NotRequired[str]
    model_used: NotRequired[str]

CONFIDENCE_THRESHOLD = 0.8

def parse_json_safe(text: str) -> dict:
    cleaned = re.sub(r"```(?:json)?\s*|\s*```", "", text).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"response": text, "confidence": 0.5}

In [4]:
def tier1_attempt(state: CascadeState) -> Command[Literal["use_tier1", "tier2"]]:
    """Fast, cheap model attempts the query first."""
    response = tier1_model.invoke(
        f"Answer this query. Also assess your confidence (0.0 to 1.0) in your answer.\n\n"
        f"Query: {state['query']}\n\n"
        f"Return JSON: {{\"response\": \"your answer\", \"confidence\": 0.85}}"
    )
    parsed = parse_json_safe(response.content)
    tier1_response = str(parsed.get("response", ""))
    tier1_confidence = float(parsed.get("confidence", 0.5))
    update = {"tier1_response": tier1_response, "tier1_confidence": tier1_confidence}
    if tier1_confidence >= CONFIDENCE_THRESHOLD:
        return Command(goto="use_tier1", update=update)
    return Command(goto="tier2", update=update)

In [5]:
def use_tier1(state: CascadeState) -> dict:
    return {"final_response": state["tier1_response"], "model_used": "gpt-4o-mini (Tier 1)"}

def tier2_attempt(state: CascadeState) -> dict:
    """Premium model handles complex queries."""
    response = tier2_model.invoke(
        f"A simpler model was not confident enough to answer this query. "
        f"Please provide a thorough, accurate answer.\n\n"
        f"Query: {state['query']}\n\n"
        f"Previous attempt (low confidence): {state.get('tier1_response', 'N/A')}"
    )
    return {
        "tier2_response": response.content,
        "final_response": response.content,
        "model_used": "gpt-4o (Tier 2)",
    }

In [6]:
graph = StateGraph(CascadeState)
graph.add_node("tier1", tier1_attempt, destinations=("use_tier1", "tier2"))
graph.add_node("use_tier1", use_tier1)
graph.add_node("tier2", tier2_attempt)

graph.add_edge(START, "tier1")
# tier1_attempt returns Command to route directly
graph.add_edge("use_tier1", END)
graph.add_edge("tier2", END)

cascade = graph.compile()

In [7]:
plot_mermaid(cascade)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	tier1(tier1)
	use_tier1(use_tier1)
	tier2(tier2)
	__end__([<p>__end__</p>]):::last
	__start__ --> tier1;
	tier1 -.-> tier2;
	tier1 -.-> use_tier1;
	tier2 --> __end__;
	use_tier1 --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [8]:
# Simple query (likely handled by Tier 1)
result = cascade.invoke({"query": "What is the capital of France?"})
print(f"Model: {result.get('model_used', 'unknown')}")
print(f"Confidence: {result.get('tier1_confidence', 'N/A')}")
print(f"Answer: {result.get('final_response', 'N/A')}")

Model: gpt-4o-mini (Tier 1)
Confidence: 1.0
Answer: Paris


In [9]:
# Complex query (likely escalated to Tier 2)
result = cascade.invoke({
    "query": "Explain the mathematical relationship between attention mechanisms and kernel methods in transformers"
})
print(f"Model: {result.get('model_used', 'unknown')}")
print(f"Answer: {result.get('final_response', 'N/A')}")

Model: gpt-4o-mini (Tier 1)
Answer: Attention mechanisms in transformers can be understood through the lens of kernel methods, particularly in how they compute similarities between inputs. In attention mechanisms, the attention score between two tokens is computed using a dot product of their corresponding query and key vectors, which can be interpreted as a similarity measure. This is akin to kernel methods, where a kernel function computes the similarity between data points in a potentially high-dimensional space. 

In the context of transformers, the attention mechanism can be seen as applying a softmax function to these similarity scores to weigh the contributions of different tokens when generating the output representation. This is similar to how kernel methods can weigh the influence of different training examples based on their similarity to a test point. 

Moreover, attention can be viewed as a form of implicit kernel computation, where the attention weights can be interpreted

In [10]:
# Streaming

stream_invoke(cascade, {"query": "What is the capital of France?"})


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'query': 'What is the capital of France?',
 'tier1_response': 'Paris',
 'tier1_confidence': 1.0,
 'final_response': 'Paris',
 'model_used': 'gpt-4o-mini (Tier 1)'}